In [ ]:
!pip install -q onnx==1.16.1 onnxruntime==1.18.1
!pip install -q onnx-tf==1.10.0
!pip install -q tensorflow==2.15.0
!pip install -q tensorflow-probability==0.23.0
print("Installed. If you see dependency warnings, they are usually harmless for this task.")
print("If a RESTART is requested, restart the kernel and run from Cell 2.")


In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models
import numpy as np

device = "cpu"   # conversion is CPU work
NUM_CLASSES = 5
CLASS_NAMES = ['Fungal','Scabies','Eczema','healthy_skin','not_skin']
STUDENT_CKPT = "MobileNet_distilled.pth"   # adjust path if needed

def build_mobilenet(n=NUM_CLASSES):
    m = models.mobilenet_v3_small(weights=None)   # no need to download pretrained for conversion
    m.classifier[3] = nn.Linear(m.classifier[3].in_features, n)
    return m

model = build_mobilenet().to(device)
model.load_state_dict(torch.load(STUDENT_CKPT, map_location=device))
model.eval()
print("PyTorch model loaded. Params:", sum(p.numel() for p in model.parameters()))


In [ ]:
import onnx

ONNX_PATH = "dermatriage_mobilenet.onnx"
dummy = torch.randn(1, 3, 224, 224, device=device)

torch.onnx.export(
    model, dummy, ONNX_PATH,
    input_names=["input"], output_names=["output"],
    opset_version=13,                 # 13 is broadly compatible with onnx-tf
    do_constant_folding=True,
    dynamic_axes=None,                # fixed batch size 1 (mobile)
)

# verify the ONNX graph is valid
onnx_model = onnx.load(ONNX_PATH)
onnx.checker.check_model(onnx_model)
print("ONNX export OK and graph is valid.")


In [ ]:
import onnxruntime as ort

test_input = np.random.randn(1, 3, 224, 224).astype(np.float32)

# PyTorch output
with torch.no_grad():
    torch_out = model(torch.from_numpy(test_input)).numpy()

# ONNX output
sess = ort.InferenceSession(ONNX_PATH)
onnx_out = sess.run(None, {"input": test_input})[0]

diff = np.abs(torch_out - onnx_out).max()
print(f"Max abs difference PyTorch vs ONNX: {diff:.6e}")
print("PASS" if diff < 1e-4 else "WARNING: difference is large — investigate before continuing")


In [ ]:
from onnx_tf.backend import prepare
import onnx

TF_DIR = "dermatriage_tf_savedmodel"
onnx_model = onnx.load(ONNX_PATH)
tf_rep = prepare(onnx_model)
tf_rep.export_graph(TF_DIR)
print(f"TensorFlow SavedModel written to {TF_DIR}/")


In [ ]:
import tensorflow as tf

# --- Float32 TFLite ---
converter = tf.lite.TFLiteConverter.from_saved_model(TF_DIR)
tflite_float = converter.convert()
with open("dermatriage_float32.tflite", "wb") as f:
    f.write(tflite_float)
print(f"Float32 TFLite: {len(tflite_float)/1e6:.2f} MB")

converter_q = tf.lite.TFLiteConverter.from_saved_model(TF_DIR)
converter_q.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_quant = converter_q.convert()
with open("dermatriage_quant.tflite", "wb") as f:
    f.write(tflite_quant)
print(f"Quantized TFLite: {len(tflite_quant)/1e6:.2f} MB")


In [ ]:
def run_tflite(tflite_path, x):
    interp = tf.lite.Interpreter(model_path=tflite_path)
    interp.allocate_tensors()
    inp = interp.get_input_details()[0]
    out = interp.get_output_details()[0]
    interp.set_tensor(inp["index"], x.astype(np.float32))
    interp.invoke()
    return interp.get_tensor(out["index"])

# same random input as before
tflite_out_f = run_tflite("dermatriage_float32.tflite", test_input)
tflite_out_q = run_tflite("dermatriage_quant.tflite", test_input)

print("Max abs diff PyTorch vs TFLite-float :", np.abs(torch_out - tflite_out_f).max())
print("Max abs diff PyTorch vs TFLite-quant :", np.abs(torch_out - tflite_out_q).max())

# Check the predicted class matches (most important for classification)
print("\nPredicted class index:")
print("  PyTorch       :", torch_out.argmax())
print("  TFLite float  :", tflite_out_f.argmax())
print("  TFLite quant  :", tflite_out_q.argmax())
print("\nIf the class indices match, the conversion is correct for classification.")


In [ ]:
from PIL import Image

IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32)

def preprocess(img_path):
    img = Image.open(img_path).convert("RGB").resize((224, 224))
    arr = np.asarray(img).astype(np.float32) / 255.0          # [0,1]
    arr = (arr - IMAGENET_MEAN) / IMAGENET_STD                 # normalize
    arr = np.transpose(arr, (2, 0, 1))                         # HWC -> CHW
    return np.expand_dims(arr, 0)                              # (1,3,224,224)


TEST_IMG = "/kaggle/working/not_skin/notskin_0001.png"

x = preprocess(TEST_IMG)
probs = tf.nn.softmax(run_tflite("dermatriage_quant.tflite", x)[0]).numpy()
pred = int(np.argmax(probs))
print(f"Prediction: {CLASS_NAMES[pred]}  ({probs[pred]:.3f})")
print("All class probs:", {CLASS_NAMES[i]: round(float(probs[i]),3) for i in range(NUM_CLASSES)})


In [ ]:
import json

deployment_info = {
    "model_file": "dermatriage_quant.tflite",
    "input_shape": [1, 3, 224, 224],
    "input_layout": "NCHW",
    "preprocessing": {
        "resize": [224, 224],
        "scale": "divide by 255 to [0,1]",
        "normalize_mean": [0.485, 0.456, 0.406],
        "normalize_std":  [0.229, 0.224, 0.225]
    },
    "class_order": CLASS_NAMES,
    "rejection_classes": ["healthy_skin", "not_skin"],
    "confidence_threshold": 0.5,
    "notes": "Apply softmax to the output logits. If argmax is a rejection class OR top prob < threshold, do not show a disease diagnosis."
}

with open("deployment_info.json", "w") as f:
    json.dump(deployment_info, f, indent=2)

print("Files to download and give to the Flutter app:")
print("  dermatriage_quant.tflite   (the model — recommended)")
print("  dermatriage_float32.tflite (fallback if quant accuracy drops)")
print("  deployment_info.json       (preprocessing + class order spec)")
print(json.dumps(deployment_info, indent=2))
